# 🏗️ Terraform — Ultra-Elaborate Mental Models

> **Every section answers four questions: WHY this exists, WHAT it is, HOW it works, WHEN to use it.**
> Real-world scenarios, ❌ before / ✅ after code, and *"Where this is seen in frameworks"* callouts.

---

**Topics**
1. The Terraform Mental Model — Infrastructure as Declarative Code
2. Architecture — Providers, State, Plan-Apply Loop
3. HCL Deep Dive — The Language of Infrastructure
4. State — The Most Important (and Dangerous) File
5. Modules — DRY Infrastructure
6. Workspaces and Remote State — Multi-Environment
7. The Plan/Apply Workflow — Safe Change Management
8. Terraform in CI/CD — Atlantis and tf-github-actions
9. Security — Least Privilege, Secret Management
10. Real-World: HashiCorp's Own Infrastructure
11. The Terraform Architect's Design Framework

---
## 1 · The Terraform Mental Model — Infrastructure as Declarative Code

### 🧠 Mental Model — *The Infrastructure Diff Engine*

> **Terraform is a diff engine for infrastructure. You declare the desired state in HCL files. Terraform compares it against the actual state (stored in a state file). It generates a plan showing exactly what will change (+ add, ~ update, - destroy). You review and approve. Then it applies. It is like a Git diff, but for cloud infrastructure.**

**WHY Terraform exists:** Before IaC:
- Infrastructure was created by clicking through AWS Console
- No record of what was created, when, or why
- Recreating an environment was a days-long archaeology project
- Two environments (staging and prod) inevitably drifted apart
- Disaster recovery: restore from backup and then manually recreate all the AWS resources around it

**The fundamental shift:**

```
❌ BEFORE (ClickOps + scripting):
  1. Log into AWS Console
  2. Click: Create VPC → Create Subnet × 6 → Create IGW → Create Route Tables...
  3. Hours later: environment created
  4. No record of what was created (unless you wrote it down)
  5. Try to create staging: start over from memory/notes
  6. Staging and prod inevitably different: 'why does this work in prod but not staging?'

✅ AFTER (Terraform IaC):
  1. Write HCL files describing desired infrastructure
  2. terraform plan  → see exactly what will be created
  3. terraform apply → infrastructure created (same as in plan, guaranteed)
  4. Git commit the HCL files (infrastructure is now versioned!)
  5. Create staging: terraform workspace new staging && terraform apply
     Identical to prod, guaranteed
  6. Disaster recovery: git clone + terraform apply
     Infrastructure recreated from code, not from memory
```

### Why Terraform Beats CloudFormation, ARM, and Pulumi

| Concern | Terraform | CloudFormation | ARM (Azure) | Pulumi |
|---|---|---|---|---|
| **Multi-cloud** | ✅ AWS + GCP + Azure + 3000+ providers | AWS only | Azure only | ✅ any (programmatic) |
| **Language** | HCL (readable, declarative) | JSON/YAML (verbose) | JSON (terrible) | Python/TypeScript/Go |
| **Community** | Huge (most popular IaC) | Large (AWS-only) | Small | Growing |
| **State management** | Explicit (tfstate) | Managed by AWS | Managed by Azure | Explicit |
| **Preview** | `terraform plan` | Change sets | Previews | `pulumi preview` |
| **Ecosystem** | Terraform Registry (thousands of modules) | CloudFormation Registry | Limited | Terraform provider compat |
| **Learning curve** | Low-Medium | Medium | High | Low (if you know Python) |

### 🌍 Real-World: How Airbnb Manages Infrastructure
Airbnb manages 500,000+ AWS resources across multiple regions using Terraform. Their approach:
- All infrastructure in a monorepo (`airbnb/infrastructure`)
- Every infrastructure change goes through a Pull Request with `terraform plan` output
- Atlantis (open source) automatically runs `terraform plan` on PRs and posts the output
- Engineers review the diff just like they review code diffs
- No one applies infrastructure manually — all changes are through CI/CD
- **Result:** Any engineer can make infrastructure changes safely, with a full audit trail

---
## 2 · Architecture — Providers, State, Plan-Apply Loop

### Component Architecture

```
Your Workspace
┌──────────────────────────────────────────────────────────────────┐
│  *.tf files           HCL configuration files (your code)        │
│  terraform.tfvars     Variable values (non-sensitive)            │
│  .terraform/          Downloaded providers (binary plugins)       │
│  terraform.tfstate    Current state (DO NOT COMMIT — sensitive!)  │
│  .terraform.lock.hcl  Provider version locks (DO commit this)    │
└──────────────────────────────────────────────────────────────────┘

terraform init
  ↓ Downloads providers from Terraform Registry
  
terraform plan
  ↓ Reads *.tf files (desired state)
  ↓ Reads tfstate (actual known state)
  ↓ Calls provider APIs to refresh state
  ↓ Diffs desired vs actual
  ↓ Outputs: what will change (+ add, ~ modify, - destroy)
  
terraform apply
  ↓ Executes the plan
  ↓ Calls cloud provider APIs to create/modify/destroy resources
  ↓ Updates tfstate with new actual state
```

### Providers — The Plugin Architecture

```hcl
# Every cloud/service you interact with needs a provider
terraform {
  required_providers {
    aws = {
      source  = "hashicorp/aws"
      version = "~> 5.0"  # ✅ pin to major version
    }
    kubernetes = {
      source  = "hashicorp/kubernetes"
      version = "~> 2.0"
    }
    datadog = {
      source  = "datadog/datadog"
      version = "~> 3.0"
    }
  }
}

# Provider configuration (credentials, region)
provider "aws" {
  region = var.aws_region
  # Credentials: from environment (AWS_ACCESS_KEY_ID) or OIDC
  # ❌ NEVER hardcode credentials here
}
```

**3,000+ providers** exist — AWS, GCP, Azure, GitHub, Datadog, PagerDuty, Cloudflare, Vault, Kubernetes, and more. If a service has an API, there's likely a Terraform provider for it.

---
## 3 · HCL Deep Dive — The Language of Infrastructure

### 🧠 Mental Model — *JSON with Superpowers*

> **HCL (HashiCorp Configuration Language) is a declarative configuration language that compiles to JSON but is much more readable. The key constructs: `resource` (a cloud thing to create), `variable` (input), `output` (exported value), `data` (read existing resource), `local` (computed value), `module` (reusable group of resources). Master these six and you can express any infrastructure.**

### Complete Production Example — AWS VPC + EKS Cluster

```hcl
# ── Variables ────────────────────────────────────────────────────────
variable "environment" {
  description = "Deployment environment (dev, staging, production)"
  type        = string
  validation {
    condition     = contains(["dev", "staging", "production"], var.environment)
    error_message = "environment must be dev, staging, or production"
  }
}

variable "cluster_version" {
  description = "Kubernetes version"
  type        = string
  default     = "1.28"
}

# ── Locals (computed values) ──────────────────────────────────────────
locals {
  name_prefix = "myapp-${var.environment}"
  common_tags = {
    Environment = var.environment
    ManagedBy   = "terraform"
    Owner       = "platform-team"
  }
}

# ── Data Sources (read existing resources) ───────────────────────────
data "aws_availability_zones" "available" {
  state = "available"
}

data "aws_eks_cluster_auth" "cluster" {
  name = module.eks.cluster_name
}

# ── Resources ─────────────────────────────────────────────────────────
resource "aws_vpc" "main" {
  cidr_block           = "10.0.0.0/16"
  enable_dns_hostnames = true
  enable_dns_support   = true

  tags = merge(local.common_tags, {
    Name = "${local.name_prefix}-vpc"
  })
}

# Dynamic subnets across AZs (using for_each)
resource "aws_subnet" "private" {
  for_each = {
    "a" = "10.0.1.0/24"
    "b" = "10.0.2.0/24"
    "c" = "10.0.3.0/24"
  }

  vpc_id            = aws_vpc.main.id
  cidr_block        = each.value
  availability_zone = "${data.aws_availability_zones.available.names[0]}"

  tags = merge(local.common_tags, {
    Name = "${local.name_prefix}-private-${each.key}"
    "kubernetes.io/role/internal-elb" = "1"  # Required for EKS LBs
  })
}

# ── Module (reusable EKS module from Terraform Registry) ─────────────
module "eks" {
  source  = "terraform-aws-modules/eks/aws"
  version = "~> 19.0"

  cluster_name    = "${local.name_prefix}-eks"
  cluster_version = var.cluster_version

  vpc_id                   = aws_vpc.main.id
  subnet_ids               = [for s in aws_subnet.private : s.id]
  cluster_endpoint_private_access = true
  cluster_endpoint_public_access  = false  # ✅ VPN only

  # EKS Managed Node Groups
  eks_managed_node_groups = {
    general = {
      instance_types = ["t3.medium"]
      min_size       = 2
      max_size       = 10
      desired_size   = 3
    }
    gpu = {
      instance_types = ["g4dn.xlarge"]
      min_size       = 0
      max_size       = 5
      desired_size   = 0
      taints = [{
        key    = "nvidia.com/gpu"
        value  = "true"
        effect = "NO_SCHEDULE"
      }]
    }
  }

  tags = local.common_tags
}

# ── Outputs ──────────────────────────────────────────────────────────
output "cluster_endpoint" {
  description = "EKS cluster API server endpoint"
  value       = module.eks.cluster_endpoint
  sensitive   = true  # ✅ hidden in logs
}

output "configure_kubectl" {
  description = "Run this command to configure kubectl"
  value       = "aws eks update-kubeconfig --region ${var.aws_region} --name ${module.eks.cluster_name}"
}
```

### Key HCL Constructs

| Construct | Purpose | Example |
|---|---|---|
| `resource` | Create cloud resource | `resource "aws_s3_bucket" "logs" {}` |
| `data` | Read existing resource | `data "aws_ami" "ubuntu" {}` |
| `variable` | Input parameter | `variable "region" { default = "us-east-1" }` |
| `output` | Export values | `output "vpc_id" { value = aws_vpc.main.id }` |
| `local` | Computed local value | `locals { name = "${var.env}-app" }` |
| `module` | Reusable group of resources | `module "vpc" { source = "./modules/vpc" }` |
| `for_each` | Create N resources from map/set | `for_each = { a = "10.0.1.0/24" }` |
| `count` | Create N identical resources | `count = 3` |
| `depends_on` | Explicit dependency | `depends_on = [aws_vpc.main]` |

---
## 4 · State — The Most Important (and Dangerous) File

### 🧠 Mental Model — *The Blueprint of What Terraform Thinks Exists*

> **Terraform state is the record of what Terraform has created. It maps your HCL resources to real cloud resource IDs. Without state, Terraform cannot know what exists — it would try to create everything again. The state file is your infrastructure's source of truth. If it's wrong, Terraform's plan is wrong. If it's lost, you either re-import everything or start over.**

### State File Contents

```json
// terraform.tfstate (simplified)
{
  "version": 4,
  "resources": [
    {
      "type": "aws_vpc",
      "name": "main",
      "provider": "provider[\"registry.terraform.io/hashicorp/aws\"]",
      "instances": [
        {
          "attributes": {
            "id": "vpc-0a1b2c3d4e5f67890",   // the REAL AWS VPC ID
            "cidr_block": "10.0.0.0/16",
            "arn": "arn:aws:ec2:us-east-1:123456789:vpc/vpc-0a1b2c3d4e5f67890"
          }
        }
      ]
    }
  ]
}
```

### Remote State — The Production Requirement

```hcl
# ❌ NEVER use local state in production:
# - Not shared with team (race conditions)
# - Not backed up (laptop dies = state lost)
# - Committed to Git = sensitive data in repo

# ✅ ALWAYS use remote state in production:
terraform {
  backend "s3" {
    bucket         = "mycompany-terraform-state"  # dedicated S3 bucket
    key            = "production/eks/terraform.tfstate"
    region         = "us-east-1"
    encrypt        = true    # ✅ AES-256 encryption at rest
    dynamodb_table = "terraform-state-lock"  # ✅ prevents concurrent applies
  }
}
```

### State Locking — The Race Condition Prevention

```
SCENARIO: Two engineers run 'terraform apply' simultaneously

Without locking:
  Engineer A reads state → plans to create VPC
  Engineer B reads state → plans to create VPC
  Engineer A creates VPC → updates state
  Engineer B creates ANOTHER VPC → updates state (overwrites A's update)
  Result: duplicate resources, corrupted state

With DynamoDB locking:
  Engineer A runs apply → acquires lock in DynamoDB
  Engineer B runs apply → "Error: state is locked by Engineer A"
  Engineer A finishes → releases lock
  Engineer B can now run
  Result: no conflicts, consistent state
```

### Cross-Stack State References

```hcl
# Read outputs from another Terraform stack's remote state
data "terraform_remote_state" "vpc" {
  backend = "s3"
  config = {
    bucket = "mycompany-terraform-state"
    key    = "production/vpc/terraform.tfstate"
    region = "us-east-1"
  }
}

# Use VPC ID from the VPC stack in the EKS stack
resource "aws_eks_cluster" "main" {
  vpc_config {
    subnet_ids = data.terraform_remote_state.vpc.outputs.private_subnet_ids
  }
}
```

In [ ]:
"""
Terraform Plan Simulator
========================
Simulates how Terraform generates a plan by comparing desired state (HCL)
with actual state (tfstate) and producing a diff.

This is the core algorithm behind 'terraform plan'.
"""
from __future__ import annotations
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Dict, List, Optional
import json

In [ ]:
class ChangeType(Enum):
    CREATE  = "create"   # + (new resource)
    UPDATE  = "update"   # ~ (modify in place)
    REPLACE = "replace"  # -/+ (destroy then create — disruptive!)
    DESTROY = "destroy"  # - (remove resource)
    NO_OP   = "no-op"    # no change


@dataclass
class ResourceChange:
    address: str          # e.g., 'aws_vpc.main'
    change_type: ChangeType
    before: Dict[str, Any] = field(default_factory=dict)
    after: Dict[str, Any]  = field(default_factory=dict)
    force_replace_reason: Optional[str] = None  # why a change forces replacement


class TerraformPlanner:
    """
    Simplified Terraform plan engine.
    Compares desired configuration with state and produces a plan.
    """

    # Attributes that force replacement (cannot be updated in-place)
    FORCE_REPLACE_ATTRS = {
        "aws_vpc": ["cidr_block"],
        "aws_subnet": ["cidr_block", "availability_zone", "vpc_id"],
        "aws_instance": ["ami", "subnet_id", "instance_type"],
        "aws_s3_bucket": ["bucket"],
        "aws_rds_cluster": ["engine", "engine_version"],  # version major changes
    }

    def plan(self,
             desired: Dict[str, Dict[str, Any]],
             state: Dict[str, Dict[str, Any]]) -> List[ResourceChange]:
        """
        desired: {resource_address: {attr: value}} — from HCL
        state:   {resource_address: {attr: value}} — from tfstate
        """
        changes = []

        # Resources in desired but not in state → CREATE
        for addr, attrs in desired.items():
            if addr not in state:
                changes.append(ResourceChange(
                    address=addr, change_type=ChangeType.CREATE,
                    after=attrs,
                ))

        # Resources in both → check for changes
        for addr in desired:
            if addr not in state:
                continue  # already handled above

            desired_attrs = desired[addr]
            state_attrs   = state[addr]
            resource_type = addr.split(".")[0]

            diffs = {k: (state_attrs.get(k), v) for k, v in desired_attrs.items()
                     if state_attrs.get(k) != v}

            if not diffs:
                changes.append(ResourceChange(
                    address=addr, change_type=ChangeType.NO_OP,
                    before=state_attrs, after=desired_attrs,
                ))
                continue

            # Check if any changed attribute forces replacement
            replace_attrs = self.FORCE_REPLACE_ATTRS.get(resource_type, [])
            forcing = [k for k in diffs if k in replace_attrs]

            if forcing:
                changes.append(ResourceChange(
                    address=addr, change_type=ChangeType.REPLACE,
                    before=state_attrs, after=desired_attrs,
                    force_replace_reason=f"forces replacement: {forcing}",
                ))
            else:
                changes.append(ResourceChange(
                    address=addr, change_type=ChangeType.UPDATE,
                    before=state_attrs, after=desired_attrs,
                ))

        # Resources in state but not in desired → DESTROY
        for addr in state:
            if addr not in desired:
                changes.append(ResourceChange(
                    address=addr, change_type=ChangeType.DESTROY,
                    before=state[addr],
                ))

        return changes

    def print_plan(self, changes: List[ResourceChange]) -> None:
        symbols = {
            ChangeType.CREATE:  "+ create",
            ChangeType.UPDATE:  "~ update",
            ChangeType.REPLACE: "-/+ replace",
            ChangeType.DESTROY: "- destroy",
            ChangeType.NO_OP:   "  no change",
        }

        print("\nTerraform will perform the following actions:")
        print("=" * 60)

        for change in changes:
            if change.change_type == ChangeType.NO_OP:
                continue
            symbol = symbols[change.change_type]
            print(f"\n  {symbol:15s}  {change.address}")

            if change.force_replace_reason:
                print(f"                ⚠️  {change.force_replace_reason}")

            # Show diff
            all_keys = set(change.before) | set(change.after)
            for key in sorted(all_keys):
                old = change.before.get(key, "(not set)")
                new = change.after.get(key, "(to be removed)")
                if old != new:
                    print(f"    ~ {key}: {json.dumps(old)} → {json.dumps(new)}")
                else:
                    print(f"      {key}: {json.dumps(new)}")

        # Summary
        creates  = sum(1 for c in changes if c.change_type == ChangeType.CREATE)
        updates  = sum(1 for c in changes if c.change_type == ChangeType.UPDATE)
        replaces = sum(1 for c in changes if c.change_type == ChangeType.REPLACE)
        destroys = sum(1 for c in changes if c.change_type == ChangeType.DESTROY)

        print("\n" + "═" * 60)
        print(f"Plan: {creates} to add, {updates} to change, {replaces} to replace, {destroys} to destroy.")
        if replaces or destroys:
            print("\n⚠️  WARNING: Destructive changes detected!")
            print("   Review carefully — these changes may cause downtime.")

In [ ]:
# Scenario: Engineer updates nginx config (in-place update)
# and changes VPC CIDR (forces replacement — DESTRUCTIVE!)

planner = TerraformPlanner()

# Current state (what Terraform knows exists in AWS)
current_state = {
    "aws_vpc.main": {
        "cidr_block": "10.0.0.0/16",
        "id": "vpc-0a1b2c3d",
        "enable_dns_hostnames": True,
    },
    "aws_security_group.nginx": {
        "name": "nginx-sg",
        "description": "nginx security group",
        "ingress_port": 80,
    },
    "aws_instance.legacy_server": {
        "ami": "ami-12345678",
        "instance_type": "t2.micro",
    },
}

# Desired state (from updated HCL files)
desired_state = {
    "aws_vpc.main": {
        "cidr_block": "10.1.0.0/16",    # ← CHANGED! Forces replacement of VPC!
        "id": "vpc-0a1b2c3d",
        "enable_dns_hostnames": True,
    },
    "aws_security_group.nginx": {
        "name": "nginx-sg",
        "description": "nginx security group v2",  # ← changed (update in-place)
        "ingress_port": 443,                         # ← changed (update in-place)
    },
    # aws_instance.legacy_server is GONE from HCL → will be destroyed
    "aws_s3_bucket.logs": {                          # ← NEW resource
        "bucket": "myapp-prod-logs",
        "versioning_enabled": True,
    },
}

changes = planner.plan(desired=desired_state, state=current_state)
planner.print_plan(changes)

---
## 5 · Modules — DRY Infrastructure

### 🧠 Mental Model — *A Terraform Module is a Function for Infrastructure*

> **A Terraform module is a group of resources with inputs (variables) and outputs. Call the same module multiple times with different inputs to create multiple identical environments. The official Terraform Registry has thousands of peer-reviewed modules — use them instead of writing raw resource blocks.**

### Module Structure

```
modules/
└── microservice/
    ├── main.tf          # Resources (ECS task, ALB target group, etc.)
    ├── variables.tf     # Input variables
    ├── outputs.tf       # Output values
    └── README.md        # Documentation

environments/
├── staging/
│   ├── main.tf
│   └── terraform.tfvars  # staging-specific values
└── production/
    ├── main.tf
    └── terraform.tfvars  # prod-specific values
```

```hcl
# environments/production/main.tf
# Using the same module for 3 different services

module "payment_service" {
  source  = "../../modules/microservice"
  
  name          = "payment-service"
  environment   = "production"
  docker_image  = "registry.company.com/payment-service:sha-a3f4b1c"
  cpu           = 512
  memory        = 1024
  min_instances = 3
  max_instances = 20
}

module "user_service" {
  source  = "../../modules/microservice"
  
  name          = "user-service"
  environment   = "production"
  docker_image  = "registry.company.com/user-service:sha-b5e7f9d"
  cpu           = 256
  memory        = 512
  min_instances = 2
  max_instances = 10
}
```

---

## 6 · Terraform in CI/CD — The GitOps Model

### 🧠 Mental Model — *Infrastructure PRs Reviewed Like Code PRs*

> **Atlantis enables the GitOps model for infrastructure: every Terraform change goes through a Pull Request. The PR shows the terraform plan output. Engineers review the plan diff before approving. Merge = apply. No one runs terraform apply locally on production — it's all automated. This gives you code review, audit trail, and blast-radius protection for infrastructure changes.**

### The Atlantis Workflow

```
1. Engineer creates PR with Terraform changes
   
2. Atlantis bot automatically:
   a. Runs terraform plan
   b. Posts the plan output as a PR comment
   
3. PR review:
   - Team members review the plan diff
   - Comment: 'atlantis plan' to re-plan after fixes
   - Comment: 'atlantis apply' to apply (requires approval first)

4. Atlantis runs terraform apply
   - Updates state in S3
   - Posts apply output to PR

5. PR is merged

This is the same workflow as a code PR — engineers treat
infrastructure changes with the same rigor as code changes.
```

---

## 7 · The Terraform Architect's Design Framework

### The 10 Commandments of Terraform at Scale

```
1. REMOTE STATE
   Always use remote state (S3 + DynamoDB locking).
   Never commit terraform.tfstate to Git.
   Encrypt state at rest.

2. MODULE EVERYTHING
   Repeat a resource pattern >1 time? Make a module.
   Use official Terraform Registry modules when they exist.
   Version pin your modules.

3. SEPARATE ENVIRONMENTS
   Separate state files per environment (not workspaces for prod).
   Test changes in staging before production.
   Different AWS accounts for dev/staging/prod (blast radius isolation).

4. PIN EVERYTHING
   Pin provider versions (~> 5.0, not > 5.0).
   Pin module versions.
   Commit .terraform.lock.hcl to Git.

5. NO SECRETS IN TERRAFORM
   Never pass secrets as variables that appear in state.
   Use data sources to read from Vault/Secrets Manager.
   Mark sensitive outputs with sensitive = true.

6. PLAN BEFORE APPLY — ALWAYS
   Never run terraform apply without reviewing the plan.
   -out flag: terraform plan -out=tfplan; terraform apply tfplan
   In CI: plan on PR, apply on merge.

7. SEPARATE STATE BY BLAST RADIUS
   VPC (rarely changes) → separate state from EKS cluster
   EKS cluster → separate state from applications
   If you change apps 10x/day and VPC 1x/quarter, don't mix them.

8. DEPENDENCY MANAGEMENT
   Use terraform_remote_state data sources for cross-stack dependencies.
   Or: use a parameter store (SSM, Consul) to share outputs between stacks.

9. TAGGING STRATEGY
   Tag EVERYTHING: Environment, Owner, ManagedBy=terraform, CostCenter.
   Tags power cost attribution, security policies, and automated cleanup.
   Use a locals block with common_tags to enforce consistency.

10. DRIFT DETECTION
    Run terraform plan on a schedule (daily) to detect drift.
    Alert when actual state differs from desired state.
    Manual console changes = drift = technical debt.
```